# 02 — XML Cache Verification

Notebook 01 already downloads and caches the verified XML (one network pass, not two).
This notebook **audits the cache**: every file exists, is well-formed XML, and contains `<infoTable>` rows — so notebook 03 never parses a cover page or an HTML error page again.

> Rule from the design discussion: notebooks never call each other — they pass data through `data/`.

In [1]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

Project root: /content/13F-Analytics
Ready.


In [2]:
from pathlib import Path
from xml.etree import ElementTree as ET
from src import config
from src.utils import load_parquet
from src.sec import looks_like_info_table

filings = load_parquet(config.FILINGS_PARQUET)
filings[["quarter", "xml_file", "local_xml_path"]]

,quarter,xml_file,local_xml_path
0,2026Q1,53405.xml,/content/13F-Analytics/data/raw_xml/2026Q1_000...
1,2025Q4,50240.xml,/content/13F-Analytics/data/raw_xml/2025Q4_000...
2,2025Q3,46994.xml,/content/13F-Analytics/data/raw_xml/2025Q3_000...
3,2025Q2,43977.xml,/content/13F-Analytics/data/raw_xml/2025Q2_000...
4,2025Q1,43981.xml,/content/13F-Analytics/data/raw_xml/2025Q1_000...
5,2024Q4,39042.xml,/content/13F-Analytics/data/raw_xml/2024Q4_000...
6,2024Q3,36917.xml,/content/13F-Analytics/data/raw_xml/2024Q3_000...
7,2024Q2,34725.xml,/content/13F-Analytics/data/raw_xml/2024Q2_000...


In [3]:
report = []
for _, f in filings.iterrows():
    p = Path(f["local_xml_path"])
    txt = p.read_text(encoding="utf-8")
    ok_xml = True
    try:
        ET.fromstring(txt)
    except ET.ParseError:
        ok_xml = False
    report.append({
        "quarter": f["quarter"],
        "file": p.name,
        "size_kb": round(p.stat().st_size / 1024, 1),
        "well_formed": ok_xml,
        "has_infoTable": looks_like_info_table(txt),
        "looks_like_html": txt.lstrip()[:6].lower().startswith("<html"),  # the old 403 symptom
    })
audit = pd.DataFrame(report)
audit

,quarter,file,size_kb,well_formed,has_infoTable,looks_like_html
0,2026Q1,2026Q1_000119312526226661.xml,44.20,True,True,False
1,2025Q4,2025Q4_000119312526054580.xml,54.10,True,True,False
2,2025Q3,2025Q3_000119312525282901.xml,56.50,True,True,False
3,2025Q2,2025Q2_000095012325008343.xml,56.00,True,True,False
4,2025Q1,2025Q1_000095012325008361.xml,2.10,True,True,False
5,2024Q4,2024Q4_000095012325002701.xml,55.10,True,True,False
6,2024Q3,2024Q3_000095012324011775.xml,59.50,True,True,False
7,2024Q2,2024Q2_000095012324008740.xml,63.50,True,True,False


In [4]:
assert audit["well_formed"].all(), "Malformed XML in cache — re-run notebook 01"
assert audit["has_infoTable"].all(), "A cover page slipped through — re-run notebook 01"
assert not audit["looks_like_html"].any(), "HTML error page cached (bad User-Agent?)"
print("cache verified: ", len(audit), "quarters ready for parsing")

cache verified:  8 quarters ready for parsing
